In [1]:
# Principal:    maria
# Role:         data_analyst
# Catalog Role: analyst_catalog_role
# Привилегии:   NAMESPACE_LIST (catalog)
#               silver: TABLE_LIST, TABLE_READ_DATA
#               gold:   TABLE_CREATE, TABLE_LIST, TABLE_READ_DATA,
#                       TABLE_WRITE_DATA
#
# Матрица доступов:
#   bronze: НЕТ ДОСТУПА
#   silver: только чтение
#   gold:   чтение + запись + создание
#
# FORBIDDEN: SELECT bronze, INSERT silver, DROP gold

In [2]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["MARIA_CLIENT_ID"]
client_secret = os.environ["MARIA_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-maria-rbac") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [3]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lakehouse    |
|spark_catalog|
+-------------+

+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [4]:
print("[ALLOWED] TABLE_LIST — lakehouse.silver")
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.silver
+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+



In [5]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.silver.customers
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers
+---+-------------------------------+--------------------------+-----------------+----------+
|id |name                           |email                     |city             |created_at|
+---+-------------------------------+--------------------------+-----------------+----------+
|6  |Антонин Бенедиктович Калашников|rfedoseeva@example.org    |к. Гудермес      |2025-04-03|
|19 |Афанасий Филиппович Аксенов    |nestorgorshkov@example.com|п. Адыгейск      |2026-01-08|
|20 |Валентина Олеговна Полякова    |reginasuvorova@example.org|ст. Городовиковск|2024-09-17|
+---+-------------------------------+--------------------------+-----------------+----------+



In [6]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.customers")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.silver.customers").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.customers
+-----------------------------+-----------------------------------------------------+-------+
|col_name                     |data_type                                            |comment|
+-----------------------------+-----------------------------------------------------+-------+
|id                           |int                                                  |NULL   |
|name                         |string                                               |NULL   |
|email                        |string                                               |NULL   |
|city                         |string                                               |NULL   |
|created_at                   |date                                                 |NULL   |
|                             |                                                     |       |
|# Metadata Columns           |                                                     |       |
|

In [7]:
print("[ALLOWED] TABLE_LIST — lakehouse.gold")
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.gold
+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [8]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category")
spark.sql("""
    SELECT
        category_name,
        month,
        total_revenue,
        order_count,
        avg_check
    FROM lakehouse.gold.mart_sales_by_category
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category
+-------------+-------+-------------+-----------+---------+
|category_name|month  |total_revenue|order_count|avg_check|
+-------------+-------+-------------+-----------+---------+
|Books        |2025-05|381074.18    |392        |972.13   |
|Clothing     |2025-05|385896.39    |427        |903.74   |
|Electronics  |2025-05|374438.50    |416        |900.09   |
+-------------+-------+-------------+-----------+---------+



In [9]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_top_customers")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.gold.mart_top_customers").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_top_customers
+-----------------------------+------------------------------------------------------------+-------+
|col_name                     |data_type                                                   |comment|
+-----------------------------+------------------------------------------------------------+-------+
|customer_id                  |int                                                         |NULL   |
|name                         |string                                                      |NULL   |
|recency_days                 |int                                                         |NULL   |
|frequency                    |bigint                                                      |NULL   |
|monetary                     |decimal(14,2)                                               |NULL   |
|segment                      |string                                                      |NULL   |
|                        

In [10]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.gold.mart_top_customers")
spark.sql("""
    INSERT INTO lakehouse.gold.mart_top_customers
    VALUES (
        99999,
        'test_customer',
        1,
        CAST(1 AS BIGINT),
        CAST(100.00 AS DECIMAL(14,2)),
        'Low'
    )
""")
print("INSERT ok")

spark.sql("""
    SELECT
        customer_id,
        name,
        recency_days,
        frequency,
        monetary,
        segment
    FROM lakehouse.gold.mart_top_customers
    WHERE customer_id = 99999
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.gold.mart_top_customers WHERE customer_id = 99999")
print("DELETE ok")

[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.gold.mart_top_customers
INSERT ok
+-----------+-------------+------------+---------+--------+-------+
|customer_id|name         |recency_days|frequency|monetary|segment|
+-----------+-------------+------------+---------+--------+-------+
|99999      |test_customer|1           |1        |100.00  |Low    |
+-----------+-------------+------------+---------+--------+-------+

DELETE ok


In [11]:
print("[ALLOWED] TABLE_CREATE — lakehouse.gold._test_gold_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.gold._test_gold_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.gold._test_gold_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

[ALLOWED] TABLE_CREATE — lakehouse.gold._test_gold_probe
CREATE ok


In [12]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.gold._test_gold_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.gold._test_gold_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.gold._test_gold_probe
    LIMIT 3
""").show(truncate=False)

Загружено строк: 100
+---+------+
|id |name  |
+---+------+
|0  |test_0|
|1  |test_1|
|2  |test_2|
+---+------+



In [13]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.gold._test_gold_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

DROP пропущен (нет прав): An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'maria' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, analyst_catalog_role]' is not authorized for op DROP_TABLE_WITHOUT_PURGE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)


In [14]:
print("[FORBIDDEN] SELECT из bronze (bronze закрыт для maria)")
try:
    spark.sql("""
        SELECT
            id,
            customer_id,
            status
        FROM lakehouse.bronze.raw_orders
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в silver.customers (silver read-only)")
try:
    spark.sql("""
        INSERT INTO lakehouse.silver.customers
        VALUES (99999, 'test', 'test@test.com', 'Moscow', CAST('2026-01-01' AS DATE))
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] DROP TABLE gold.mart_top_customers (нет TABLE_DROP)")
try:
    spark.sql("DROP TABLE lakehouse.gold.mart_top_customers")
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

[FORBIDDEN] SELECT из bronze (bronze закрыт для maria)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'maria' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, analyst_catalog_role]' is not authorized for op LOAD_TABLE
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] INSERT в silver.customers (silver read-only)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'maria' with activated PrincipalRoles '[data_analyst]' and activated grants via '[data_analyst, analyst_catalog_role]' is not authorized for op ADD_TABLE_SNAPSHOT
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] DROP TABLE gold.mart_top_customers (нет TABLE_DROP)
[ОЖИДАЕМО] Доступ запрещён: An err